In [2]:
# --- [ 1단계: 모든 흔적 지우기 및 con 재생성 ] ---
import os
import duckdb
from dotenv import load_dotenv , find_dotenv
load_dotenv(find_dotenv())
endpoint = os.getenv('MINIO_ENDPOINT')
access_key = os.getenv('MINIO_ACCESS_KEY')
secret_key = os.getenv('MINIO_SECRET_KEY')
# 아예 새 그릇을 꺼냅니다 (기존 con이 있다면 닫힘)
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
# --- [ 2단계: 'Secret' 저장소에 직접 각인하기 ] ---
# 기존에 꼬여있던 낱개 SET 방식이 아니라, 하나의 '비밀번호 팩'을 만드는 방식입니다.
con.execute(f"""
    CREATE OR REPLACE SECRET my_minio_secret (
        TYPE S3,
        KEY_ID '{access_key}',
        SECRET '{secret_key}',
        ENDPOINT '{endpoint}',
        URL_STYLE 'path',
        USE_SSL 'false',
        REGION 'us-east-1'
    );
""")
# --- [ 3단계: %sql 매직 커맨드 초기화 ] ---
%load_ext sql
%sql con 
# 잘 되는지 확인용 (이게 에러 안 나면 성공입니다)
print("✅ [강제 초기화 완료] 이제 찌꺼기 없는 깨끗한 상태입니다. 쿼리를 날려보세요!")

✅ [강제 초기화 완료] 이제 찌꺼기 없는 깨끗한 상태입니다. 쿼리를 날려보세요!


In [14]:
path = "s3://petroleum-project/national_avg/*/data.parquet"
test_df = con.sql(f"""
SELECT 
cast (part_dt as string) as part_dt 
, prodcd
, price
FROM '{path}' 
where prodcd = 'B027'
and part_dt >= 20080415

"""

).df()
    

display(test_df.head())
    


,part_dt,PRODCD,PRICE
0,20080415,B027,1681.33
1,20080416,B027,1692.15
2,20080417,B027,1686.56
3,20080418,B027,1689.68
4,20080419,B027,1692.91


In [13]:
%%sql


select 
*
from 's3://petroleum-project/national_avg/*/data.parquet'
where part_dt >= '20080415'
and diff is not null 



Running query in 'DuckDBPyConnection'

part_dt,PRODCD,PRODNM,PRICE,DIFF,collect_time
20260124,B034,고급휘발유,1930.51,-1.41,2026-01-24 23:07:33
20260124,B027,휘발유,1692.0,-0.91,2026-01-24 23:07:33
20260124,D047,자동차용경유,1584.9,-1.41,2026-01-24 23:07:33
20260124,C004,실내등유,1318.76,-0.7,2026-01-24 23:07:33
20260124,K015,자동차용부탄,998.04,-0.07,2026-01-24 23:07:33
20260125,B034,고급휘발유,1930.51,0.09,2026-01-25 01:00:10
20260125,B027,휘발유,1692.02,-0.01,2026-01-25 01:00:10
20260125,D047,자동차용경유,1584.96,-0.07,2026-01-25 01:00:10
20260125,C004,실내등유,1318.84,-0.12,2026-01-25 01:00:10
20260125,K015,자동차용부탄,997.83,-0.32,2026-01-25 01:00:10


In [16]:
import plotly.express as px

fig = px.line(test_df , x = 'part_dt' , y = 'PRICE' , title = '18년간 휘발유 유가 추이')
fig.update_xaxes(rangeslider_visible=True)
fig.show()

In [3]:
%%sql

select *
from 'E:\ds_project\coordinates\whitelist\nationwide_master_grid_katec.parquet'
limit 100;

Running query in 'DuckDBPyConnection'

katec_x,katec_y,lat,lon
102380.89,59005.51,33.082243,124.8123253
110880.89,59005.51,33.0845367,124.9032482
119380.89,59005.51,33.0867642,124.9941813
127880.89,59005.51,33.0889254,125.0851242
136380.89,59005.51,33.0910205,125.1760767
144880.89,59005.51,33.0930493,125.2670385
153380.89,59005.51,33.0950119,125.3580092
161880.89,59005.51,33.0969082,125.4489887
170380.89,59005.51,33.0987381,125.5399765
178880.89,59005.51,33.1005017,125.6309723
